Preprocessing


In [2]:
import os
import librosa
import soundfile as sf

# Folder containing user folders U001-U023
root_folder = r'User_Voice'

# Target sampling rate
SR = 16000

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in os.listdir(user_path):
        if file.endswith('.wav'):
            file_path = os.path.join(user_path, file)
            y, sr = librosa.load(file_path, sr=SR, mono=True)
            # Trim leading/trailing silence
            y, _ = librosa.effects.trim(y)
            # Normalize volume
            y = y / max(abs(y))
            # Save preprocessed audio
            sf.write(file_path, y, SR)
print("Preprocessing complete!")


Preprocessing complete!


Feature Extraction


In [3]:
import numpy as np

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SR)
    
    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfccs = np.mean(mfccs.T, axis=0)  # shape (40,)
    
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma = np.mean(chroma.T, axis=0)
    
    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr)
    mel = np.mean(mel.T, axis=0)
    
    # Prosodic features
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch = pitches[magnitudes > np.median(magnitudes)]
    pitch_mean = np.mean(pitch) if len(pitch) > 0 else 0
    pitch_std = np.std(pitch) if len(pitch) > 0 else 0
    
    energy = np.mean(y**2)
    
    features = np.hstack([mfccs, chroma, mel[:40], pitch_mean, pitch_std, energy])
    return features  # final vector


Dataset Preparation

In [6]:
import pandas as pd

X = []
y_user = []
y_question = []

for user_folder in sorted(os.listdir(root_folder)):
    user_path = os.path.join(root_folder, user_folder)
    if not os.path.isdir(user_path):
        continue
    for file in sorted(os.listdir(user_path)):
        if file.endswith('.wav'):
            file_path = os.path.join(user_path, file)
            features = extract_features(file_path)
            X.append(features)
            
            # User label
            y_user.append(user_folder)
            # Question label based on filename: 01.wav -> 1
            q_num = int(file.split('.')[0])
            y_question.append(q_num)

X = np.array(X)
y_user = np.array(y_user)
y_question = np.array(y_question)

print("Dataset ready!")
print("X shape:", X.shape)
print("User labels shape:", y_user.shape)
print("Question labels shape:", y_question.shape)


Dataset ready!
X shape: (230, 95)
User labels shape: (230,)
Question labels shape: (230,)


In [7]:
from sklearn.model_selection import train_test_split

# Split for speaker recognition (users)
X_train, X_test, y_train_user, y_test_user = train_test_split(
    X, y_user, test_size=0.2, stratify=y_user, random_state=42
)

# Optional: split for question classification
_, _, y_train_q, y_test_q = train_test_split(
    X, y_question, test_size=0.2, stratify=y_question, random_state=42
)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Speaker classification
clf_user = RandomForestClassifier(n_estimators=200, random_state=42)
clf_user.fit(X_train, y_train_user)

y_pred_user = clf_user.predict(X_test)
acc_user = accuracy_score(y_test_user, y_pred_user)
print("Speaker recognition accuracy:", acc_user)

# Question classification (cognitive response)
clf_q = RandomForestClassifier(n_estimators=200, random_state=42)
clf_q.fit(X_train, y_train_q)

y_pred_q = clf_q.predict(X_test)
acc_q = accuracy_score(y_test_q, y_pred_q)
print("Question classification accuracy:", acc_q)


Speaker recognition accuracy: 1.0
Question classification accuracy: 0.021739130434782608


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class VoiceDataset(Dataset):
    def __init__(self, X, y_user, y_question):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_user = y_user
        self.y_question = y_question
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y_user[idx], self.y_question[idx]

# Define simple MLP
class VoiceMLP(nn.Module):
    def __init__(self, input_dim, num_users, num_questions):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )
        self.user_head = nn.Linear(128, num_users)
        self.q_head = nn.Linear(128, num_questions)
    
    def forward(self, x):
        x = self.shared(x)
        return self.user_head(x), self.q_head(x)

